# Lab 3 - Data Quality & Cleaning

In [1]:
import sys
!"{sys.executable}" -m pip install pandas
!"{sys.executable}" -m pip install numpy
!"{sys.executable}" -m pip install matplotlib
!"{sys.executable}" -m pip install seaborn
!"{sys.executable}" -m pip install scipy
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy

## Part 1: Data Cleaning

In this lab, we will explore common problems found with datasets, and investigate ways to fix these problems.

First, download messy_transactions.csv from the Moodle. This is a synthetic dataset representing company transactions.

In [3]:
messy_transactions = pd.read_csv('messy_transactions.csv')
messy_transactions.head()

,Transaction ID,Transaction Date,Item,Quantity,Price Per Unit,Transaction Total,Customer Name,Customer ID
0,TXN-105397,2026-01-08,Printer,6.0,656.2,3937.20,Mr Louis Taylor,C00447
1,TXN-103077,2024-05-07,Monitor,7.0,469.09,3283.63,Laura Atkinson-Baker,C00005
2,TXN-106050,2026-04-16,Mouse,10.0,1341.67,NaN,Rachel Goodwin,C00146
3,TXN-106502,2024-11-01,Laptop,-2.0,698.88,4892.16,Glen Sharpe,C00909
4,TXN-103601,2025-04-30,Monitor,1.0,408.33,408.33,Mrs Marilyn Powell,C00667


This dataset is intentionally dirty. Let's try and understand our data a little more:

In [4]:
messy_transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 10300 entries, 0 to 10299
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Transaction ID     10274 non-null  str    
 1   Transaction Date   10266 non-null  str    
 2   Item               10260 non-null  str    
 3   Quantity           10259 non-null  float64
 4   Price Per Unit     10268 non-null  str    
 5   Transaction Total  10270 non-null  float64
 6   Customer Name      10267 non-null  str    
 7   Customer ID        10020 non-null  str    
dtypes: float64(2), str(6)
memory usage: 643.9 KB


We can see here that we hae 10,300 entries in our dataframe. messy_transactions.info() tells us, for each column (attribute), how many non-null values we have. Looking through our columns, all appear to contain null values, ranging from around 26 for _Transaction ID_, to almost 300 for _Customer ID_.

messy_transactions.info() also tells us the data types of our columns, and the memory usage of our data frame.

Q1: Other than the null values, can you spot anything concerning in the output of messy_transactions.info()? Hint: look at data types.

Let's take another look at the data:

In [5]:
messy_transactions.head()

,Transaction ID,Transaction Date,Item,Quantity,Price Per Unit,Transaction Total,Customer Name,Customer ID
0,TXN-105397,2026-01-08,Printer,6.0,656.2,3937.20,Mr Louis Taylor,C00447
1,TXN-103077,2024-05-07,Monitor,7.0,469.09,3283.63,Laura Atkinson-Baker,C00005
2,TXN-106050,2026-04-16,Mouse,10.0,1341.67,NaN,Rachel Goodwin,C00146
3,TXN-106502,2024-11-01,Laptop,-2.0,698.88,4892.16,Glen Sharpe,C00909
4,TXN-103601,2025-04-30,Monitor,1.0,408.33,408.33,Mrs Marilyn Powell,C00667


From first glance, this dataset looks to have satisfactory quality. It represents sales data in a store. We can see transaction ID, information about the specific order, item purchases, transaction total, details about the customer, etc.

Is there consistency in the dataset? Well, what sort of consistency would we expect? For any given customer ID, this should always return the same customer name. Let's check if that's the case.

In [6]:
counts = messy_transactions.groupby("Customer ID")["Customer Name"].nunique()

bad_ids = counts[counts > 1].index

messy_transactions_duplicate_customer_IDs = messy_transactions[messy_transactions["Customer ID"].isin(bad_ids)].sort_values(by="Customer ID")

messy_transactions_duplicate_customer_IDs.head()

print(messy_transactions_duplicate_customer_IDs)

messy_transactions_duplicate_customer_IDs.to_csv("messy_transactions_duplicate_customer_IDs.csv")

     Transaction ID Transaction Date             Item  Quantity  \
4247     TXN-102647       2025-02-16          Printer      13.0   
4752     TXN-107772       2023-08-25           Webcam      13.0   
9502     TXN-104621       2024-05-22         Keyboard       3.0   
3544     TXN-109380       2025-06-19          Headset      18.0   
576      TXN-105577       2024-11-19            Mouse       4.0   
...             ...              ...              ...       ...   
533      TXN-103231       2024-08-11            Mouse      13.0   
1957     TXN-104055       2024-03-03  Docking Station       9.0   
7770     TXN-103339       2024-01-04           Router      15.0   
7684     TXN-104571       2025-10-09          Monitor      14.0   
9517     TXN-106493       2024-03-01           Webcam       6.0   

     Price Per Unit  Transaction Total            Customer Name Customer ID  
4247        1339.01           17407.13               Paul Brown      000123  
4752         252.94            3288.22 

Look at the dataset (you may want to open it in a text editor). We can see that for some of these names, the inconsistency is arising due to incorrect capitalisation（大写字母） or inappropriate whitespace. Let's try to fix these issues using the code below:
1. Remove double whitespace
2. Fix capitalisation.

In [7]:
print("Before cleaning: " + messy_transactions.iloc[3303]["Customer Name"])
messy_transactions["Customer Name"] = messy_transactions["Customer Name"].str.strip().str.replace(r"\s+", " ", regex=True)
#"\s+" returns one or more spaces. The .str.replace() replaces any occurrence of one or more whitespaces with a single whitespace. .str.strip() replaces leading or trailing spaces.
print("After cleaning: " + messy_transactions.iloc[3303]["Customer Name"])
#Check an example with multiple whitespaces.

print("Before cleaning: " + messy_transactions.iloc[7652]["Customer Name"])
messy_transactions["Customer Name"] = messy_transactions["Customer Name"].str.title()
print("After cleaning: " + messy_transactions.iloc[7652]["Customer Name"])

Before cleaning:  Beverley Yates
After cleaning: Beverley Yates
Before cleaning: ALEX FULLER
After cleaning: Alex Fuller


We should check for exact duplicates. Let's do this using the code below, and output our results to a CSV:

In [8]:
messy_transactions[messy_transactions.duplicated(keep=False)].to_csv("duplicates.csv")

Please note that without "keep=False", Pandas will consider a duplicate to be an object which has appeared in the dataset previously. For example, if a row exists three times in the dataset, only the latter two instances will be considered "duplicates", rather than all three.

We can see that there are many duplicates! Checking some of these records in the CSV using your favourite text editor confirms this. Let's remove them:

In [9]:
messy_transactions = messy_transactions.drop_duplicates()

In [10]:
messy_transactions.head()
messy_transactions.to_csv("messy_transactions_duplicates_removed.csv")

Q3: The lab sheet so far has provided a short introduction into the kinds of intial checks we'd perform on a dataset to fix some common data quality issues. Now, your job is to find, and fix, more quality issues with this data! Your company has provided you with the following specification:

a. The "Transaction ID" must be in the format "TXN-XXXXXX", where XXXXXX is some six-digit number.  
b. The "Transaction ID" must be unique.  
c. The "Customer ID" must be unique.  
d. "Quantity" should be non-negative (0+). If a negative value is found, it should be convereted to a positive value*.  
e. "Price per unit" should be non-negative (0+). If a negative value is found, it should be flagged.  
f. "Transaction Total" must equal "Quantity" $\times$ "Price per unit".

The above list is not exhaustive - there are other issues in this data which you should try to identify and fix! If it is not feasible to fix this data in the lab, you should propose a solution that could be taken by a real company in this position.

*This is, of course, dangerous! We can assume the company found a bug in their code and is confident that changing the sign keeps the data valid!

Extension Question: Your company is concerned that there are some _near duplicates_. That is, some rows which are almost identical to other rows, but are not *identical*. Your company wants these records highlighted for manual review. Create some code to identify these near duplicates.

In [12]:
# 只比较 Customer ID 和 Customer Name 两列，其他列的差异不影响判断
# keep=False：标记所有匹配记录，包括第一次出现的记录
near_duplicates = messy_transactions[
    messy_transactions.duplicated(
        subset=["Customer ID", "Customer Name"],
        keep=False
    )
]

# 按照客户 ID 排序，方便人工检查
near_duplicates = near_duplicates.sort_values(by="Customer ID")

# 在 Jupyter 中显示
display(near_duplicates)

# 导出 CSV 文件，供人工审核
near_duplicates.to_csv("near_duplicates.csv", index=False)

,Transaction ID,Transaction Date,Item,Quantity,Price Per Unit,Transaction Total,Customer Name,Customer ID
7253,TXN-105160,2026-04-02,Docking Station,10.0,1133.75,11337.50,Elizabeth Ward,12345
5944,TXN-103515,2023-09-08,Webcam,9.0,1139.85,10258.65,Edward Williams,12345
2726,TXN-105629,2025-12-18,Headset,12.0,1025.41,12304.92,Edward Williams,12345
2857,TXN-101643,2023-12-09,Router,4.0,325.78,1303.12,Elizabeth Ward,12345
315,TXN-107661,2024-04-01,Headset,3.0,1488.54,4465.62,Jordan Harrison,C00000
...,...,...,...,...,...,...,...,...
9489,TXN-106882,31/02/2025,Laptop,-5.0,548.22,9999.00,Glenn Clark,NaN
9597,TXN-104363,2025-12-16,Webcam,10.0,590.7,5907.00,Carolyn Walters-Hill,NaN
9887,TXN-101081,31/02/2025,Keyboard,-5.0,1416.5,9999.00,Clive Humphries,NaN
10004,TXN-107022,2026-01-09,USB Drive,13.0,539.35,7011.55,Laura Smith,NaN


## Part 2: Data Preprocessing

Preparing data for processing is not just about fixing issues. We may also need to prepare the data not by correcting flaws, but simply by modifying our data to put it in a more appropriate format for processing, without losing the integrity or validity of the data.

For this part of the lab, we will use the California Housing dataset. It is available on the Moodle. Due to computational resource limits, we are using a version with only 1000 data objects.

In [13]:
housing = pd.read_csv('cal_housing_1000.csv')
housing.head()

,longitude,latitude,housingMedianAge,totalRooms,totalBedrooms,population,households,medianIncome,medianHouseValue
0,-117.35,33.23,4,1837,287,934,277,3.8958,189800
1,-118.98,35.39,29,607,177,476,143,1.1875,50700
2,-122.33,37.98,3,2850,544,1024,515,6.0115,175000
3,-121.90,37.45,18,4900,814,2984,758,6.6176,276200
4,-118.17,33.98,41,756,195,873,212,2.7321,156000


Let's say we want to compute the Euclidean distance between each object. We can do this like so:

In [ ]:
# 导入距离计算函数 pdist 和矩阵转换函数 squareform
from scipy.spatial.distance import pdist, squareform

# 计算 housing 中任意两行之间的欧氏距离，返回一维压缩距离数组
distances = pdist(housing.values, metric="euclidean")

# 将一维距离数组转换为 N×N 的对称距离矩阵
distance_matrix = squareform(distances)

# 将 NumPy 距离矩阵转换为 DataFrame，并保留原始 housing 的行索引
distance_matrix = pd.DataFrame(distance_matrix, index=housing.index, columns=housing.index)

# 输出所有样本之间的距离矩阵
print(distance_matrix)

# 输出 housing 的形状：(行数, 列数)
print(housing.shape)

               0              1              2              3    \
0         0.000000  139106.302372   14839.036857   86481.522887   
1    139106.302372       0.000000  124322.544791  225556.542956   
2     14839.036857  124322.544791       0.000000  101240.388555   
3     86481.522887  225556.542956  101240.388555       0.000000   
4     33817.544893  105300.878648   19121.263950  120292.771620   
..             ...            ...            ...            ...   
995   66602.519054   72524.515822   51803.882273  153033.017693   
996  279300.544177  418404.078180  294100.500539  192928.404847   
997   23602.695408  162704.606996   38421.205407   62917.180958   
998  177712.538590  316818.480648  192502.984548   91321.612342   
999   57601.056915   81514.487136   42811.456082  144042.913026   

               4              5              6              7    \
0     33817.544893  310208.072342   63800.393501  281601.924594   
1    105300.878648  449313.988265   75312.301123  420700.1641

Let's take a look at the closest 10 houses to house 0.

In [ ]:
# distance_matrix[0]：获取距离矩阵中列标签为 0 的一列，即各房屋到房屋 0 的距离
# .nsmallest(11)：选取距离最小的 11 个值（通常包含房屋 0 自身）
# .index：提取这些距离对应的房屋索引，而不是距离数值
closest_houses = distance_matrix[0].nsmallest(11).index

Although we wanted the closest 10, we have pulled the 11 closest houses. This is because the most similar house to any given house, is itself! Let's have a look at house 0 and its 10 closest houses.

In [16]:
print(housing.iloc[closest_houses])

     longitude   latitude   housingMedianAge   totalRooms   totalBedrooms  \
0      -117.35      33.23                  4         1837             287   
463    -122.07      37.95                 39         2199             388   
536    -118.21      34.14                 25         1908             628   
903    -121.27      38.66                 15         2642             520   
699    -122.82      38.64                 29         2176             385   
723    -119.19      34.16                 34         2610             466   
620    -118.27      33.82                 37          943             218   
890    -118.25      34.09                 52         3142             765   
853    -122.26      37.83                 52         1470             330   
287    -118.13      33.90                 38         1475             269   
737    -121.29      38.36                 17         2193             386   

      population   households   medianIncome   medianHouseValue  
0        

From a quick glance, we an see that the values in the medianHouseValue column are all fairly similar to each other, whereas some other columns (e.g. longitude, latitude, totalRooms, population, ...) differ quite significantly! Let's look a little more under-the-hood to try and see what is happening here:

In [17]:
# 获取 housing 中第一套房屋的全部特征，作为距离计算的参照对象
house_0 = housing.iloc[0]

# 将每套房屋的各个特征减去 house_0 对应的特征，再将差值平方
# 结果仍是一个 DataFrame，形状与 housing 相同
squared_diff_all_houses = (housing - house_0) ** 2

# 按列求和，得到每个特征在所有房屋中的平方差总和
global_distance_sum = squared_diff_all_houses.sum()

# 将所有特征的平方差总和再次相加，得到整体平方差总量
grand_total_sum = global_distance_sum.sum()

# 计算每个特征的平方差占总平方差的百分比
global_percentage_share = (global_distance_sum / grand_total_sum) * 100

# 按贡献百分比从大到小排序并打印
print(global_percentage_share.sort_values(ascending=False))


 medianHouseValue    9.995391e+01
 totalRooms          3.393017e-02
 population          9.371569e-03
 totalBedrooms       1.533264e-03
 households          1.246872e-03
 housingMedianAge    6.068632e-06
 latitude            7.940523e-08
longitude            6.629143e-08
 medianIncome        2.589642e-08
dtype: float64


We can see above the medianHouseValue alone represents around 99% of the distance between house 0 and the other houses! The other attributes, barely contribute anything to our distance calculation.

We would like to think that when calculating the distance between houses, we want to consider a variety of different attributes, not just medianHouseValue. Let's use Min-Max Normalisation.

In [ ]:
housing_normalised = (housing - housing.min())/(housing.max() - housing.min())
#Apply Min-Max Normalisaiton.
print(housing_normalised)

Note that we don't need to do this manually. We can use the MinMaxScaler method built into Scikitlearn if we wanted to.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

housing_normalised = pd.DataFrame(scaler.fit_transform(housing), columns=housing.columns)

print(housing_normalised)

Now, let's calculate distances again:

In [ ]:
from scipy.spatial.distance import pdist, squareform

distances = pdist(housing_normalised.values, metric="euclidean")

distance_matrix = squareform(distances)

distance_matrix = pd.DataFrame(distance_matrix, index=housing_normalised.index, columns=housing_normalised.index)

closest_houses = distance_matrix[0].nsmallest(11).index

print(housing.iloc[closest_houses])

And, if we again calculate the percentage that each attribute is contributing to our distance calculation:

In [ ]:
house_0 = housing_normalised.iloc[0]
#Let's focus on the first house (house 0) in the data.

squared_diff_all_houses = (housing_normalised - house_0) ** 2
#Create a data frame containing the squared distance between house 0 and the rest of the houses, per attribute.

global_distance_sum = squared_diff_all_houses.sum()
#Take the sum of all the distances between house 0 and the other houses, at a per attribute level.

grand_total_sum = global_distance_sum.sum()
#Sum over all attributes.

global_percentage_share = (global_distance_sum / grand_total_sum) * 100
#Calculate the percentage share of each attribute to our distance calculation.

print(global_percentage_share.sort_values(ascending=False))

We have reduced the impact of medianHouseValue significantly! However, some attributes (totalBedrooms, households, totalRooms and population) still contribute little to our distance calcluation. Why do you think this is? In this case, it is because there are some outliers heavily influencing our normalisation. For example, the quartiles of the population attribute are 0.05, 0.07 and 0.1, despite being normalised to between 0 and 1.